# 1.3 자기상관 실습 — MSFT 수익률과 제곱 수익률의 시간 종속성

> **학습 목표**
> - ACF/PACF를 직접 그리고 해석한다
> - Ljung-Box 검정으로 "자기상관 존재 여부" 를 결정한다
> - 금융 수익률 vs 제곱 수익률의 ACF 패턴 차이를 눈으로 확인한다 (변동성 클러스터링)
> - 이 결과로부터 시퀀스 모델의 `lookback window T` 를 몇으로 잡을지 근거를 마련한다

**데이터**: `../../black_litterman/data/panels/MSFT.csv` (1.2 실습과 동일)


## Step 0 — 환경 설정 (한글 폰트 + 라이브러리)

한글 시각화를 위해 CLAUDE.md 지침의 폰트 블록을 먼저 실행합니다.
이 단계에서는 `statsmodels` 의 ACF/PACF/Ljung-Box 함수를 import 합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform

# CLAUDE.md 지침의 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    import koreanize_matplotlib  # pip install koreanize-matplotlib --break-system-packages
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# statsmodels 자기상관 관련 함수
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

pd.set_option('display.float_format', '{:.5f}'.format)
print('환경 준비 완료')


## Step 1 — 데이터 로드 및 기본 점검

1.2 실습과 동일한 MSFT 데이터를 불러오되, 이번에는 **제곱 수익률** 컬럼까지 추가합니다.
제곱 수익률은 변동성 대리 변수로, 금융 시계열 분석의 핵심 피처입니다.


In [ ]:
CSV = '../../black_litterman/data/panels/MSFT.csv'
df = pd.read_csv(CSV, parse_dates=['date']).set_index('date').sort_index()

# 제곱 수익률 컬럼 추가 (변동성 대리 변수)
df['sq_log_return_1d'] = df['log_return_1d'] ** 2

# 절댓값 수익률도 추가 (fat-tail 영향을 덜 받는 대안)
df['abs_log_return_1d'] = df['log_return_1d'].abs()

print('Shape:', df.shape)
print('Period:', df.index.min().date(), '~', df.index.max().date())
print('Columns (이번 실습에 사용):', ['close', 'log_return_1d', 'sq_log_return_1d', 'abs_log_return_1d'])
df[['close', 'log_return_1d', 'sq_log_return_1d', 'abs_log_return_1d']].head()


## Step 2 — close의 ACF/PACF (비정상 시계열 기준점)

비정상 시계열인 `close` 의 ACF는 **매우 천천히** 감소합니다.
이것이 "추세가 있는 시계열" 의 전형적 패턴입니다. 정상화(차분)가 필요하다는 시각적 증거가 됩니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(df['close'].dropna(), lags=40, ax=axes[0])
axes[0].set_title('close ACF — 천천히 감소 (비정상 전형)')

plot_pacf(df['close'].dropna(), lags=40, ax=axes[1], method='ywm')
axes[1].set_title('close PACF — lag 1에서 1에 가까운 값 (랜덤워크)')

plt.tight_layout()
plt.show()


**관찰 포인트**

- `close` ACF: lag 40까지도 0.9 이상 유지 → **강한 추세** 가 있다는 증거
- `close` PACF: lag 1만 ≈ 1, 이후 급락 → **랜덤워크(Random Walk)** 의 전형적 형태

> 랜덤워크: $X_t = X_{t-1} + \epsilon_t$ (오늘 = 어제 + 노이즈). 가격 모델의 기본 가정.

이런 시계열로는 어떤 시계열 분석도 의미가 없습니다. 반드시 차분하여 수익률로 변환 후 작업합니다.


## Step 3 — log_return_1d의 ACF/PACF (수익률 = 거의 백색잡음)

이제 정상화된 **일별 로그 수익률** 을 봅시다. 금융 이론이 예측하는 결과는:
- ACF 거의 0 (효율적 시장 가설)
- 유의한 시차가 있어도 **경제적으로 이용 가능한 크기는 아님**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(df['log_return_1d'].dropna(), lags=40, ax=axes[0])
axes[0].set_title('log_return_1d ACF — 대부분 0 근처 (백색잡음 유사)')

plot_pacf(df['log_return_1d'].dropna(), lags=40, ax=axes[1], method='ywm')
axes[1].set_title('log_return_1d PACF — 대부분 유의하지 않음')

plt.tight_layout()
plt.show()


**관찰 포인트**

- 신뢰구간(파란 음영) 밖으로 삐져나오는 막대가 **거의 없음**
- 일부 시차에서 살짝 유의해 보여도 값 자체가 0.05 수준 → 경제적 의미 없음
- **"어제 수익률로 오늘 수익률을 맞출 수 없다"** 라는 효율적 시장 가설의 실증적 근거


### Step 3.5 — 수치로 다시 확인

시각화로 본 결과를 정확한 수치로 확인합니다. `acf()` 는 값과 신뢰구간을 모두 반환합니다.


In [ ]:
# 처음 10개 시차의 ACF 값을 수치로 비교
n = df['log_return_1d'].dropna().shape[0]
ci_95 = 1.96 / np.sqrt(n)
print(f'표본 크기 n = {n}, 95% 신뢰구간 = ±{ci_95:.4f}')
print()

ret_acf_vals = acf(df['log_return_1d'].dropna(), nlags=10, fft=True)
close_acf_vals = acf(df['close'].dropna(), nlags=10, fft=True)

print('=== ACF 값 비교 (lag 0~10) ===')
print(f'{"lag":>5} | {"close":>10} | {"log_return_1d":>15} | {"log_return 유의?":>20}')
print('-' * 65)
for h in range(11):
    sig = '예' if abs(ret_acf_vals[h]) > ci_95 else '아니오'
    print(f'{h:>5} | {close_acf_vals[h]:>10.4f} | {ret_acf_vals[h]:>15.4f} | {sig:>20}')


**해석**

- `close` 는 모든 시차에서 거의 1 → 극도로 강한 시간 종속
- `log_return_1d` 는 대부분 ±0.04 이내 → **유의하긴 해도 매우 작음**
- 참고: 표본이 2000개를 넘으면 신뢰구간이 작아져서 아주 작은 값도 "통계적 유의"로 나오지만, 경제적 활용 가치는 별개 문제입니다.


## Step 4 — **제곱 수익률** 의 ACF (금융 시계열의 핵심)

이번 실습의 하이라이트입니다.
`sq_log_return_1d = log_return_1d ** 2` 의 ACF는 **완전히 다른 그림** 을 보여줍니다.
이것이 **변동성 클러스터링(Volatility Clustering)** 의 수치적 증거입니다.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 수익률 자기상관 (참고)
plot_acf(df['log_return_1d'].dropna(), lags=40, ax=axes[0, 0])
axes[0, 0].set_title('log_return_1d ACF (참고: 거의 0)')

plot_acf(df['log_return_1d'].dropna(), lags=40, ax=axes[0, 1])
axes[0, 1].set_title('log_return_1d ACF (동일)')
axes[0, 1].set_ylim(-0.1, 0.4)  # 같은 스케일로 비교

# 제곱 수익률 자기상관 (핵심)
plot_acf(df['sq_log_return_1d'].dropna(), lags=40, ax=axes[1, 0])
axes[1, 0].set_title('sq_log_return_1d ACF — 강한 지속적 자기상관!')
axes[1, 0].set_ylim(-0.1, 0.4)

# 절댓값 수익률 (더 안정적 지표)
plot_acf(df['abs_log_return_1d'].dropna(), lags=40, ax=axes[1, 1])
axes[1, 1].set_title('abs_log_return_1d ACF — 더 뚜렷한 지속 자기상관')
axes[1, 1].set_ylim(-0.1, 0.4)

plt.tight_layout()
plt.show()


**이것이 이 챕터의 핵심 결과입니다.**

| 시계열 | ACF 패턴 | 경제적 의미 |
|---|---|---|
| `log_return_1d` | 거의 0 | 수익률 방향은 예측 불가 (EMH) |
| `sq_log_return_1d` | 수십 시차까지 양의 값 | **변동성은 예측 가능** |
| `abs_log_return_1d` | 제곱보다 더 뚜렷 | fat-tail 영향을 덜 받는 변동성 지표 |

> **변동성 클러스터링**: 큰 움직임(상승이든 하락이든) 다음에 큰 움직임이 오고, 작은 움직임 다음에 작은 움직임이 오는 경향.
>
> 이것이 **GARCH 모델 계열** 이 존재하는 이유이고, 우리 프로젝트에서 `vol_20d_ann` 피처가 의미있는 이유입니다.


## Step 5 — Ljung-Box 검정으로 확정

ACF 시각화는 개별 lag별 판단이지만, Ljung-Box는 **lag 1~m 을 묶어서 전체적으로** 자기상관 존재 여부를 검정합니다.

| | H₀ (귀무가설) | H₁ (대립가설) |
|---|---|---|
| Ljung-Box | 모든 시차에서 자기상관 = 0 | 어떤 시차에서 자기상관 ≠ 0 |

**해석 규칙**: p-value < 0.05 → 자기상관 있음 (H₀ 기각)


In [ ]:
# 각 시계열에 대해 lag=[10, 20, 40] 에서 검정
targets = ['log_return_1d', 'sq_log_return_1d', 'abs_log_return_1d', 'close']
lags_to_test = [10, 20, 40]

lb_results = []
for col in targets:
    series = df[col].dropna()
    result = acorr_ljungbox(series, lags=lags_to_test, return_df=True)
    result = result.copy()
    result.insert(0, 'series', col)
    result.insert(1, 'lag', result.index)
    lb_results.append(result.reset_index(drop=True))

lb_all = pd.concat(lb_results, ignore_index=True)
lb_all['유의?(0.05)'] = lb_all['lb_pvalue'].apply(lambda p: '자기상관 있음' if p < 0.05 else '자기상관 없음')
print('=== Ljung-Box 검정 결과 ===')
print(lb_all[['series', 'lag', 'lb_stat', 'lb_pvalue', '유의?(0.05)']].to_string(index=False))


**예상되는 결과 패턴**

- `close`: 모든 lag에서 p ≈ 0 → 강한 자기상관 (비정상)
- `log_return_1d`: p 값이 작더라도 ACF 자체가 작음 → **통계적 유의 vs 경제적 의미 주의**
- `sq_log_return_1d`: p ≈ 0 → **변동성 클러스터링 확정**
- `abs_log_return_1d`: p ≈ 0 → 동일

> **주의**: 샘플이 크면 (n > 2000) Ljung-Box는 작은 자기상관도 "유의"로 판정합니다.
> 그래서 **시각적 크기 + p-value** 를 같이 봐야 의사결정이 제대로 됩니다.


## Step 6 — Lookback Window T 결정 근거 만들기

2주차에서 실제 T 값을 정할 때 쓸 근거 데이터를 여기서 계산합니다.

**원칙**
1. 수익률 ACF가 0을 향해 완전히 떨어진 시점 → 수익률 정보의 유효 범위
2. 제곱 수익률 ACF가 0에 수렴하는 시점 → 변동성 정보의 유효 범위 (보통 더 김)
3. 두 값 중 **더 큰 값** 을 T로 잡는 것이 합리적 (변동성 정보까지 포착)


In [ ]:
# lag 1~120 까지 계산해서 유의 시차 찾기
max_lag = 120

ret_series = df['log_return_1d'].dropna()
sq_series = df['sq_log_return_1d'].dropna()

n_ret = len(ret_series)
ci_ret = 1.96 / np.sqrt(n_ret)
n_sq = len(sq_series)
ci_sq = 1.96 / np.sqrt(n_sq)

acf_ret = acf(ret_series, nlags=max_lag, fft=True)
acf_sq = acf(sq_series, nlags=max_lag, fft=True)

# 유의 시차 찾기 (신뢰구간 밖)
sig_ret_lags = [h for h in range(1, max_lag + 1) if abs(acf_ret[h]) > ci_ret]
sig_sq_lags = [h for h in range(1, max_lag + 1) if abs(acf_sq[h]) > ci_sq]

print(f'log_return_1d 유의 시차 (|ACF|>±{ci_ret:.4f}):')
print(f'  개수: {len(sig_ret_lags)}개 / {max_lag}개')
print(f'  시차들: {sig_ret_lags[:20]}{" ..." if len(sig_ret_lags) > 20 else ""}')
print(f'  최대 유의 시차: {max(sig_ret_lags) if sig_ret_lags else "없음"}')
print()
print(f'sq_log_return_1d 유의 시차 (|ACF|>±{ci_sq:.4f}):')
print(f'  개수: {len(sig_sq_lags)}개 / {max_lag}개')
print(f'  시차들: {sig_sq_lags[:20]}{" ..." if len(sig_sq_lags) > 20 else ""}')
print(f'  최대 유의 시차: {max(sig_sq_lags) if sig_sq_lags else "없음"}')


In [ ]:
# lag별 ACF 값 그래프로 비교
fig, ax = plt.subplots(figsize=(14, 5))

lags = range(1, max_lag + 1)
ax.plot(lags, acf_ret[1:], label='log_return_1d (수익률)', marker='o', markersize=3, linewidth=1)
ax.plot(lags, acf_sq[1:], label='sq_log_return_1d (제곱 = 변동성)', marker='s', markersize=3, linewidth=1)

# 신뢰구간
ax.axhline(ci_ret, color='gray', linestyle='--', alpha=0.5, label=f'95% CI (±{ci_ret:.3f})')
ax.axhline(-ci_ret, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linewidth=0.5)

# 추천 T 지점 표시
for T_candidate in [20, 60, 120]:
    ax.axvline(T_candidate, color='red', alpha=0.3, linestyle=':')
    ax.text(T_candidate, 0.35, f'T={T_candidate}', rotation=0, color='red',
            ha='center', fontsize=9)

ax.set_xlabel('Lag (일)')
ax.set_ylabel('ACF')
ax.set_title('Lookback Window T 결정 근거: 수익률 vs 제곱 수익률 ACF 비교')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**해석 가이드**

- 수익률 ACF는 거의 0축에 붙어 있음 → 수익률 자체의 "기억"은 짧음
- 제곱 수익률 ACF는 수십 시차까지 양의 값 유지 → 변동성의 "기억"은 김
- T=20 은 단기 변동성, T=60 은 중기(현재 프로젝트의 mom_60d 와 일치), T=120 은 장기
- **권장 시작점: T = 60** (프로젝트의 기존 feature engineering과 정합)

### 2주차에 사용할 T 후보 결정

```python
# 권장 설정 (2주차에 SequenceDataset 구현 시 사용)
LOOKBACK = 60  # 프로젝트의 mom_60d와 정합, 제곱 수익률 자기상관도 이 정도면 상당히 반영됨
HORIZON = 21   # 기존 fwd_ret_21d와 동일
```


## Step 7 — 결론 및 2주차 연결

### 오늘 확인한 사실

| 사실 | 증거 | 결론 |
|---|---|---|
| 수익률 자기상관은 거의 0 | ACF 모든 시차에서 ±0.05 이내 | 개별 수익률 예측 불가 |
| 제곱 수익률 자기상관은 크고 지속적 | ACF lag 수십 개까지 양의 값 | **변동성은 예측 가능** |
| Ljung-Box로 두 현상 모두 통계적 유의 | p-value 거의 0 | 통계적 확정 |

### 시퀀스 모델 설계에 대한 함의

1. **입력에 수익률만 넣으면 부족**: 수익률 자체의 시간 종속이 약하므로, GRU가 학습할 신호가 약함
2. **변동성 관련 파생 피처 또는 raw 제곱 수익률을 함께**: 변동성 클러스터링이 강한 신호원
3. **T = 60 이 합리적 시작점**: 기존 프로젝트의 `mom_60d` 와 정합, 변동성 자기상관도 상당 부분 포함

### 다음 단계

- **1.4 시계열 분해** 로 진행 (추세 + 계절성 + 잔차)
- 1주차 마무리 산출물: `실습_MSFT_정상성분석.ipynb` (선택, 1.1~1.4 통합 노트북)
- 2주차에서 이 결과를 근거로 `SequenceDataset` 구현
